In [1]:
FILE = 'scratch/Smaalenenes_1903_Excel_Correction.xlsx'   # <-- set to your Excel file name
OUT  = 'validation_report.csv'

In [2]:
import pandas as pd, re
from collections import Counter

df = pd.read_excel(FILE, dtype=str).fillna('').reset_index(drop=True)
cols = list(df.columns)
def has(*c): return all(x in cols for x in c)

issues, skipped = [], []
def flag(check, i, detail):
    r = df.iloc[i] if i is not None else {}
    issues.append({'check': check, 'excel_row': (i + 2) if i is not None else '',
                   'gaards_no': r.get('gaards_no', ''), 'brugs_no': r.get('brugs_no', ''),
                   'detail': detail})

mk = pd.to_numeric(df['mark'], errors='coerce').fillna(0) if 'mark' in cols else None
oe = pd.to_numeric(df['ore'],  errors='coerce').fillna(0) if 'ore'  in cols else None

if 'eier_bruker' in cols:
    eb = df['eier_bruker'].str.strip()
    is_total = eb.str.contains('overf', case=False, na=False)
    is_sogn  = eb.str.contains(r'\bsogn\b', case=False, na=False) & ~eb.str.contains(r'\d')
    is_data  = ~is_total & ~is_sogn
else:
    is_total = pd.Series(False, index=df.index)
    is_data  = pd.Series(True, index=df.index)

# type safety: numeric columns clean
num_re, ore_re = re.compile(r'^\d+$'), re.compile(r'^\d{1,2}$')
if has('mark', 'ore'):
    for i in df.index[is_data]:
        m, o = df.at[i, 'mark'].strip(), df.at[i, 'ore'].strip()
        if m and not num_re.match(m): flag('type', i, f'mark not numeric: {m!r}')
        if o and not ore_re.match(o): flag('type', i, f'ore not 2-digit: {o!r}')
        if m and not o: flag('type', i, 'mark without ore')
else: skipped.append('type (no mark/ore)')

# balance: independent page-sum recompute (ore units)
if has('mark', 'ore') and is_total.any():
    tot = list(df.index[is_total])
    for a, c in zip(tot, tot[1:]):
        sd = df.iloc[a + 1:c][is_data.iloc[a + 1:c].values]
        s = int(pd.to_numeric(sd['mark'], errors='coerce').fillna(0).sum()) * 100 + \
            int(pd.to_numeric(sd['ore'],  errors='coerce').fillna(0).sum())
        va = int(round(mk[a])) * 100 + int(round(oe[a])); vc = int(round(mk[c])) * 100 + int(round(oe[c]))
        if vc - va != s: flag('balance', c, f'delta {(vc - va - s)/100:.2f} (total {vc/100:.2f}, prev {va/100:.2f}, page {s/100:.2f})')
else: skipped.append('balance (no mark/ore or no totals)')

# words_in_CD: cols C and D are duplicate gaards/brugs (numeric); flag stray words
LET = re.compile(r'[A-Za-zÆØÅæøå]')
for pos in (2, 3):
    if pos < len(cols):
        col = cols[pos]
        v = df.loc[is_data, col].str.strip(); ne = v[v != '']
        if len(ne) and ne.str.match(r'^\d+$').mean() > 0.5:
            for i in ne.index:
                if LET.search(df.at[i, col]): flag('words_in_CD', i, f'{col} (col {chr(65 + pos)}) words: {df.at[i, col]!r}')

rep = pd.DataFrame(issues, columns=['check', 'excel_row', 'gaards_no', 'brugs_no', 'detail'])
rep.to_csv(OUT, index=False, encoding='utf-8-sig')

print('columns:', cols)
if skipped: print('skipped checks:', skipped)
print(rep['check'].value_counts().to_string() if len(rep) else 'no issues')
print(f'\n{len(rep)} issues -> {OUT}')

columns: ['gaards_no', 'brugs_no', 'gaards_no_raw', 'brugs_no_raw', 'gaardens_navn', 'brugets_navn', 'eier_bruker', 'mark', 'ore', 'anmerkn', 'postanstalt', 'check']
check
words_in_CD    166
balance         64

230 issues -> validation_report.csv


In [3]:
import pandas as pd
report = pd.read_csv(OUT)
print(f'{len(report)} issues')
report

230 issues


,check,excel_row,gaards_no,brugs_no,detail
0,balance,751,NaN,NaN,"delta 2.00 (total 2243.18, prev 2146.80, page ..."
1,balance,871,NaN,NaN,"delta -2454.09 (total 191.09, prev 2454.09, pa..."
2,balance,1489,NaN,NaN,"delta -1440.64 (total 151.41, prev 1440.64, pa..."
3,balance,1885,NaN,NaN,"delta 1.00 (total 1656.84, prev 1538.88, page ..."
4,balance,1928,NaN,NaN,"delta -0.19 (total 1804.51, prev 1656.84, page..."
...,...,...,...,...,...
225,words_in_CD,15153,88.0,1.0,brugs_no_raw (col D) words: 'Haugen.....'
226,words_in_CD,15316,NaN,NaN,brugs_no_raw (col D) words: 'Lundum søndre Lun...
227,words_in_CD,15322,NaN,2.0,brugs_no_raw (col D) words: '<math>^{2}</math>'
228,words_in_CD,15362,NaN,2.0,brugs_no_raw (col D) words: 'Ulsrød.....'


In [5]:
import pandas as pd
report = pd.read_csv(OUT)
n_pages = max(int(is_total.sum()) - 1, 0)
n_bad   = int((report['check'] == 'balance').sum())
n_cells = int((df.loc[is_data, 'mark'].str.strip() != '').sum() +
              (df.loc[is_data, 'ore'].str.strip()  != '').sum())   # filled mark+ore values
print(f'Balance errors: {n_bad} of {n_pages} pages  ({n_bad / n_pages:.1%})' if n_pages else 'no pages')
print(f'Cell level:     >= {n_bad} of {n_cells} skyld cells  ({n_bad / n_cells:.2%})' if n_cells else 'no cells')

Balance errors: 64 of 706 pages  (9.1%)
Cell level:     >= 64 of 21371 skyld cells  (0.30%)
